# Make the reduced SKADI tutorial data

This notebook creates the small McStas file used by the SKADI user-guide notebook. It keeps every 50th event across the complete detector, removes histogram outputs and their stale plotting metadata, and retains only the component positions and rotations required to reconstruct the detector geometry. The final `h5repack` step removes the vacated HDF5 space.

The original simulation is not downloaded because of its size. Place `mccode.h5` from `all_banks_1e8_mpi4_sample10` in `SKADI_example_data` before running the notebook. The destination must not already exist.

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import h5py

## Reduction function

Events are sampled using their global index rather than sampling each detector group independently. This produces exactly $\lfloor N / 50 \rfloor$ events while preserving their original detector groups and metadata.

In [ ]:
def shrink_skadi_mcstas(
    source: Path, destination: Path, *, factor: int
) -> dict[str, int | Path]:
    """Subsample events and retain only geometry needed by the SKADI loader."""
    if factor < 1:
        raise ValueError("factor must be at least 1")
    if destination.exists():
        raise FileExistsError(f"Destination already exists: {destination}")
    h5repack = shutil.which("h5repack")
    if h5repack is None:
        raise RuntimeError("h5repack must be available on PATH")

    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(
        prefix="shrink-skadi-mcstas-", dir=destination.parent
    ) as tmpdir:
        working_copy = Path(tmpdir) / source.name
        repacked = Path(tmpdir) / f"repacked-{source.name}"
        shutil.copyfile(source, working_copy)

        original_event_count = 0
        retained_event_count = 0
        removed_histogram_count = 0
        removed_component_count = 0
        with h5py.File(working_copy, "r+") as file:
            data_groups = file["entry1/data"]
            event_groups = [
                group
                for group in data_groups.values()
                if isinstance(group, h5py.Group) and "events" in group
            ]
            required_component_names = {
                group.attrs["component"].decode()
                if isinstance(group.attrs["component"], bytes)
                else group.attrs["component"]
                for group in event_groups
            }
            required_component_names.update(("sourceESS", "sample_position"))

            components = file["entry1/instrument/components"]
            component_names = {
                name.split("_", maxsplit=1)[-1]
                for name, group in components.items()
                if isinstance(group, h5py.Group)
            }
            missing = required_component_names - component_names
            if missing:
                raise ValueError(
                    "Required instrument components are missing: "
                    + ", ".join(sorted(missing))
                )

            for name in list(components):
                component_name = name.split("_", maxsplit=1)[-1]
                component = components[name]
                if "output" in component and "BINS" in component["output"]:
                    removed_histogram_count += 1
                if component_name not in required_component_names:
                    del components[name]
                    removed_component_count += 1
                    continue

                for child in list(component):
                    if child not in {"Position", "Rotation"}:
                        del component[child]

            for group in event_groups:
                events = group["events"]
                event_count = events.shape[0]
                start = (factor - 1 - original_event_count % factor) % factor
                retained = events[start::factor]
                attrs = dict(events.attrs)
                del group["events"]
                events = group.create_dataset("events", data=retained)
                events.attrs.update(attrs)
                for attr in (
                    "signal",
                    "statistics",
                    "target",
                    "type",
                    "values",
                    "xylimits",
                ):
                    if attr in group.attrs:
                        del group.attrs[attr]

                original_event_count += event_count
                retained_event_count += retained.shape[0]

        expected_event_count = original_event_count // factor
        if retained_event_count != expected_event_count:
            raise RuntimeError(
                f"Expected {expected_event_count} retained events, got "
                f"{retained_event_count}"
            )

        subprocess.run(  # noqa: S603
            [h5repack, str(working_copy), str(repacked)],
            check=True,
        )
        shutil.move(repacked, destination)

    return {
        "source_events": original_event_count,
        "retained_events": retained_event_count,
        "removed_histograms": removed_histogram_count,
        "removed_components": removed_component_count,
        "output_bytes": destination.stat().st_size,
        "destination": destination,
    }

## Create the tutorial file

The path setup works when Jupyter is launched from either the repository root, the `esssans` package directory, or this `tools` directory.

In [ ]:
working_directory = Path.cwd()
if (working_directory / "packages/esssans").is_dir():
    package_root = working_directory / "packages/esssans"
elif working_directory.name == "tools":
    package_root = working_directory.parent
else:
    package_root = working_directory

data_directory = package_root / "SKADI_example_data"
source = data_directory / "all_banks_1e8_mpi4_sample10/mccode.h5"
destination = data_directory / "skadi_mcstas_1e8_sample10_1_of_50.h5"

In [ ]:
shrink_skadi_mcstas(source, destination, factor=50)